# AF2MTS1 — seed-42 kill gate
Training-only P3/P4/P5 scaffold. Validation and exported checkpoint use native AF2 inference. Target: Macro ≥90.50%, Bottom-3 ≥84.50%, Worst not lower than AF2CTRL. Test is unavailable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib, json, os, shutil, subprocess, sys, tarfile, time, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection')
BRANCH='codex/af2-multilevel-training-scaffold'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    completed=subprocess.run(clone)
    if completed.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('Git clone gagal tiga kali.')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('SETUP SELESAI:',BRANCH)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
AF2_REL='experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt'
CTRL_REL='experiments/faruq-v3-af2-complement-v1/val_reports/AF2CTRL_seed42_result.json'
ARCHIVE_REL='bundles/faruq-development-v3-grouped.tar'
PROJECT=resolve_drive_project_root(required_relative_paths=(ARCHIVE_REL,AF2_REL,CTRL_REL))
ARCHIVE=require_project_artifact(PROJECT,ARCHIVE_REL)
AF2=require_project_artifact(PROJECT,AF2_REL)
CONTROL=require_project_artifact(PROJECT,CTRL_REL)
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'data.yaml').is_file() and not (DATA/'test').exists()
OUTPUT=PROJECT/'experiments/faruq-v3-af2-multilevel-scaffold-v1'
STATIC=OUTPUT/'static_audit.json'; OUTPUT.mkdir(parents=True,exist_ok=True)
print('PROJECT:',PROJECT); print('AF2:',AF2); print('OUTPUT:',OUTPUT)

In [ ]:
audit_command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_scaffold_audit','--af2-checkpoint',str(AF2),'--output',str(STATIC),'--device','0']
AUDIT_LOG=OUTPUT/'static_audit_run.log'
print('STATIC AUDIT:', ' '.join(audit_command),flush=True)
with AUDIT_LOG.open('w',encoding='utf-8') as stream:
    completed=subprocess.run(audit_command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
print('\n'.join(AUDIT_LOG.read_text(errors='replace').splitlines()[-200:]))
if completed.returncode!=0:
    if STATIC.is_file(): print('STATIC JSON:',STATIC.read_text(errors='replace'))
    raise RuntimeError(f'Static audit gagal: {completed.returncode}; log={AUDIT_LOG}')
audit=json.loads(STATIC.read_text()); print('METHOD:',audit['equivalence_method']); print('GATES:',audit['gates']); print('DECISION:',audit['decision'])
assert audit['decision']=='PASS','STOP: static audit gagal; training tidak dijalankan.'

In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_scaffold_arm','--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--af2-checkpoint',str(AF2),'--static-audit',str(STATIC),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
LOG=OUTPUT/'AF2MTS1_seed42_run.log'
print('START/RESUME AF2MTS1 | log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    last_reported=-1
    while process.poll() is None:
        csv=OUTPUT/'AF2MTS1/AF2MTS1_seed42/results.csv'
        epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epochs!=last_reported and (epochs==0 or epochs % 5 == 0): print(f'AF2MTS1: {epochs}/30 epoch tercatat',flush=True); last_reported=epochs
        time.sleep(30)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-160:])); raise RuntimeError(f'AF2MTS1 gagal: {process.returncode}')
print('TRAINING DAN EXPORT SELESAI')

In [ ]:
CANDIDATE=OUTPUT/'val_reports/AF2MTS1_seed42_result.json'
DECISION=OUTPUT/'val_reports/af2mts1_seed42_decision.json'
decision_command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_scaffold_decision','--control-result',str(CONTROL),'--candidate-result',str(CANDIDATE),'--output',str(DECISION)]
subprocess.run(decision_command,cwd=REPO,check=True)
result=json.loads(DECISION.read_text())
print('VALUES:',result['values']); print('DELTAS:',result['deltas']); print('CRITERIA:',result['criteria']); print('DECISION:',result['decision']); print('NEXT:',result['next'])
print('Kirim output ini. Jangan membuka test.')